In [2]:

import os

from kxor_code.algorithms.base_alg_step import ProblemRecord
folder_path = "data/results/"
files = os.listdir(folder_path)


for file_name in files[:100]:
    record = ProblemRecord.load_compact(folder_path + file_name)
    print(record.instance)
    print(record.load_compact(folder_path + file_name).step_history)

KXORInstance(n=16, k=2, m=293, scopes=array([[ 1, 12],
       [ 6, 15],
       [ 1, 11],
       [ 1,  8],
       [11, 12],
       [ 8, 11],
       [ 7, 12],
       [ 2,  5],
       [10, 11],
       [ 8, 12],
       [ 3,  6],
       [ 8, 14],
       [12, 13],
       [ 2,  9],
       [ 5, 10],
       [ 7, 14],
       [10, 12],
       [ 2,  5],
       [ 0,  7],
       [ 2, 11],
       [11, 13],
       [ 6, 14],
       [ 5, 13],
       [ 7, 12],
       [ 2,  6],
       [ 5,  7],
       [ 8, 10],
       [ 2,  6],
       [ 9, 11],
       [ 4, 12],
       [ 6, 12],
       [ 5, 14],
       [ 3, 10],
       [ 2, 13],
       [ 0, 12],
       [11, 12],
       [ 7, 11],
       [ 8, 11],
       [ 7,  9],
       [ 2,  3],
       [ 6, 10],
       [ 7, 13],
       [ 1, 12],
       [ 9, 15],
       [ 1,  8],
       [ 4,  9],
       [ 5,  6],
       [ 3,  4],
       [13, 14],
       [ 3, 13],
       [ 4, 12],
       [ 4,  6],
       [ 1,  8],
       [11, 15],
       [ 6, 15],
       [ 5, 12],
       [ 0

In [6]:
record_1 = ProblemRecord.load_compact(
"data/small_result_test_set/problem_n16_k2_m111_rho0.8_kappa1.0_ell2.pkl.npz"
)
record_2 = ProblemRecord.load_compact(
"data/small_result_test_set/problem_n16_k2_m152_rho0.8_kappa0.75_ell2.pkl.npz"
)
record_3 = ProblemRecord.load_compact(
    "data/small_result_test_set/problem_n19_k2_m201_rho0.8_kappa0.75_ell2.pkl.npz"
)
print(record_1.step_history, record_2.step_history, record_3.step_history)

[StepStats(step_name='ComputeKikuchiStep', version='0.1', problem_id='test_problem', started_at=1453.535323515, completed_at=1453.553091386, failed=False, additional_data={}), StepStats(step_name='ThresholdStep', version='0.1', problem_id='test_problem', started_at=1453.553255704, completed_at=1453.553470528, failed=False, additional_data={}), StepStats(step_name='ClassicalEigenvaluesStep', version='0.1', problem_id='test_problem', started_at=1453.553550027, completed_at=1453.60191497, failed=False, additional_data={})] [StepStats(step_name='ComputeKikuchiStep', version='0.1', problem_id='test_problem', started_at=1452.465645808, completed_at=1452.490024788, failed=False, additional_data={}), StepStats(step_name='ThresholdStep', version='0.1', problem_id='test_problem', started_at=1452.490181072, completed_at=1452.490415161, failed=False, additional_data={}), StepStats(step_name='ClassicalEigenvaluesStep', version='0.1', problem_id='test_problem', started_at=1452.490485403, completed_a

In [ ]:
"data/kxor_dataset/kxor_instance_n19_k2_m370_rho0.8.npz"

In [2]:
from kxor_code.algorithms.base_alg_step import ProblemRecord
record = ProblemRecord.load_compact("data/small_result_test_set/problem_n16_k2_m111_rho0.8_kappa1.0_ell2.pkl.npz")

In [3]:
print("z_hat in fields?", "z_hat" in record.fields)
print("ground truth z?", record.instance.z is not None)
print("fields:", sorted(record.fields.keys()))

z_hat in fields? False
ground truth z? True
fields: ['eigenvalues', 'eigenvectors', 'ell', 'failure_prob', 'kappa', 'kikuchi_matrix', 'num_eigenvalues', 'rho', 'threshold']


In [4]:
import importlib

import kxor_code.algorithms.key_extraction_step as kes

# If the notebook kernel already imported this module before a code change, reload it so we
# pick up the current KeyExtractionStep implementation.
importlib.reload(kes)
KeyExtractionStep = kes.KeyExtractionStep

print("VALID_STAGE1_BACKENDS:", KeyExtractionStep.VALID_STAGE1_BACKENDS)

# Ensure `record` exists even if you run this cell first.
try:
    record  # noqa: F821
except NameError:
    from kxor_code.algorithms.base_alg_step import ProblemRecord
    record = ProblemRecord.load_compact(
        "data/small_result_test_set/problem_n16_k2_m111_rho0.8_kappa1.0_ell2.pkl.npz"
    )

# For saved compact records, this backend reuses the stored eigenvector instead of trying to
# simulate the full stage-1 PennyLane circuit (which is infeasible for these m).
stage1_backend = "precomputed_eigenvector"

# Stage-2: use circuit backend only if PennyLane is available in this kernel; otherwise fall back
# to the classical eigensolver so the notebook still runs in minimal environments.
try:
    import pennylane as qml  # noqa: F401
    stage2_backend = None  # default (circuit)
    print("PennyLane found; using circuit eigensolver for stage 2.")
except ModuleNotFoundError:
    print("PennyLane not found; using classical eigensolver for stage 2.")
    stage2_backend = "classical_eigsh"
print("Finished try-except")

step = KeyExtractionStep(stage1_backend=stage1_backend, stage2_backend=stage2_backend, evaluate=True)
stats = step.execute(record)

print("failed?", stats.failed)
if stats.failed:
    print("error:", stats.additional_data.get("error"))
else:
    z_hat = record.get_field("z_hat")
    print("z_hat head:", z_hat[:10])
    for k in ["eval_advantage", "eval_hamming_frac", "eval_correlation"]:
        if k in stats.additional_data:
            print(k, stats.additional_data[k])

2026-01-19 10:14:35,268 - KeyExtractionStep - INFO - Starting step 'KeyExtractionStep' (version 0.1) on problem test_problem (fields={'ell': 2, 'kappa': 1.0, 'failure_prob': 0.75, 'rho': 0.8, 'num_eigenvalues': 3, 'threshold': 25.900000000000002, 'eigenvalues': array([ 9.8518219 , 10.04754886, 15.10823464]), 'eigenvectors': array([[ 0.05147333, -0.07200079, -0.03336621],
       [ 0.01145171, -0.04484193,  0.08312735],
       [ 0.0284337 , -0.02921812,  0.06907073],
       [ 0.0083887 ,  0.0004227 , -0.07836627],
       [ 0.06735053,  0.04888092, -0.07292712],
       [-0.01473826, -0.02560494,  0.08345873],
       [ 0.00500057, -0.09677516,  0.05522757],
       [ 0.07193559,  0.04834811, -0.07439721],
       [ 0.10563015,  0.01606481,  0.04677193],
       [ 0.041542  , -0.02842542,  0.07800662],
       [ 0.01990001, -0.10242221,  0.08298886],
       [ 0.00471601,  0.08205147, -0.0733847 ],
       [-0.02181148, -0.11844968,  0.05918365],
       [ 0.01590791,  0.04584939,  0.06123619],
  

VALID_STAGE1_BACKENDS: {'precomputed_eigenvector', 'pipeline_qnode', 'quartic_step_adapter'}
PennyLane found; using circuit eigensolver for stage 2.
Finished try-except


2026-01-19 10:14:39,094 - KeyExtractionStep - INFO - Eval: adv=0.3514, hamming=2/16 (0.125), corr=0.750
2026-01-19 10:14:39,096 - KeyExtractionStep - INFO - Key extraction complete: x_hat=1
2026-01-19 10:14:39,134 - KeyExtractionStep - INFO - Finished step 'KeyExtractionStep' in 3.862s (success: True, problem=test_problem)


failed? False
z_hat head: [-1  1 -1 -1  1  1  1 -1  1 -1]
eval_advantage 0.35135135135135137
eval_hamming_frac 0.125
eval_correlation 0.75


In [ ]:
# Second run to test classical Eigensolver in stage 2

import importlib

import kxor_code.algorithms.key_extraction_step as kes

# If the notebook kernel already imported this module before a code change, reload it so we
# pick up the current KeyExtractionStep implementation.
importlib.reload(kes)
KeyExtractionStep = kes.KeyExtractionStep

print("VALID_STAGE1_BACKENDS:", KeyExtractionStep.VALID_STAGE1_BACKENDS)

# Ensure `record` exists even if you run this cell first.
try:
    record  # noqa: F821
except NameError:
    from kxor_code.algorithms.base_alg_step import ProblemRecord
    record = ProblemRecord.load_compact(
        "data/small_result_test_set/problem_n16_k2_m111_rho0.8_kappa1.0_ell2.pkl.npz"
    )

# For saved compact records, this backend reuses the stored eigenvector instead of trying to
# simulate the full stage-1 PennyLane circuit (which is infeasible for these m).
stage1_backend = "precomputed_eigenvector"

# Stage-2: use circuit backend only if PennyLane is available in this kernel; otherwise fall back
# to the classical eigensolver so the notebook still runs in minimal environments.

print("PennyLane not found; using classical eigensolver for stage 2.")
stage2_backend = "classical_eigsh"
print("Finished try-except")

step = KeyExtractionStep(stage1_backend=stage1_backend, stage2_backend=stage2_backend, evaluate=True)
stats = step.execute(record)

print("failed?", stats.failed)
if stats.failed:
    print("error:", stats.additional_data.get("error"))
else:
    z_hat = record.get_field("z_hat")
    print("z_hat head:", z_hat[:10])
    for k in ["eval_advantage", "eval_hamming_frac", "eval_correlation"]:
        if k in stats.additional_data:
            print(k, stats.additional_data[k])